In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import polars as pl
import numpy as np
import datetime
import os
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# Dual setup for both local and kaggle (of competition if exists)
kaggle_path = '/kaggle/input/competitions/hbaac-round2/'
train_path = kaggle_path + 'train.csv' if os.path.exists(kaggle_path) else 'train.csv'
sample_sub_path = kaggle_path + 'sample_submission.csv' if os.path.exists(kaggle_path) else 'sample_submission.csv'

split_date = datetime.date(2025, 7, 9)

print("Phase 1: Cleaning data")
train_df = pl.read_csv(train_path, infer_schema_length=0)
if 'Stt' in train_df.columns:
    train_df = train_df.drop('Stt')

numeric_cols = ['Quantity', 'UnitPrice', 'SalesAmount', 'Unit Cost', 'Cost Amount']
for col in numeric_cols:
    if col in train_df.columns:
        train_df = train_df.with_columns(
            pl.col(col).str.replace_all(',', '')
                       .str.strip_chars()
                       .cast(pl.Float32, strict=False)
        )

for col in ['UnitPrice', 'Unit Cost', 'Cost Amount']:
    if col in train_df.columns:
        train_df = train_df.with_columns(
            pl.col(col).fill_null(pl.col(col).median().over('ItemCode')).fill_null(0)
        )

train_df = train_df.with_columns([
    pl.col('Quantity').fill_null(0),
    pl.col('SalesAmount').fill_null(0)
])

# Standardize Dates
train_df = train_df.with_columns(
    pl.col('Date').str.to_date('%Y-%m-%d', strict=False)
).drop_nulls(subset=['Date', 'ItemCode'])

# Calculate WRMSSE Weights (Profit-based)
train_df = train_df.with_columns(
    (pl.col('SalesAmount') - pl.col('Cost Amount')).alias('Profit')
)

weights_df = train_df.group_by('ItemCode').agg(
    pl.col('Profit').sum().alias('TotalProfit')
).with_columns(
    # If profit is negative --> = 0
    pl.when(pl.col('TotalProfit') <= 0).then(0).otherwise(pl.col('TotalProfit')).alias('AdjustedProfit')
)

total_profit = weights_df['AdjustedProfit'].sum()
weights_df = weights_df.with_columns(
    (pl.col('AdjustedProfit') / total_profit).alias('Weight')
)
print(f"Total SKUs with positive weight: {weights_df.filter(pl.col('Weight') > 0).shape[0]}")

In [ ]:
print("Phase 2: Building Master Date Grid")

# Separate actual sales from returns (negative quantity)
daily_sales = train_df.group_by(['ItemCode', 'Date']).agg([
    pl.col('Quantity').sum().alias('Raw_Quantity'),
    pl.when(pl.col('Quantity') < 0).then(pl.col('Quantity').abs()).otherwise(0).sum().alias('Returns_Feature')
]).with_columns(
    pl.when(pl.col('Raw_Quantity') < 0).then(0).otherwise(pl.col('Raw_Quantity')).alias('Target_Quantity')
)

# Cross join all SKUs with all dates in history to spawn "0 sale" days
unique_skus = train_df.select('ItemCode').unique()
date_range = pl.date_range(train_df['Date'].min(), train_df['Date'].max(), interval="1d", eager=True).alias("Date").to_frame()
grid_df = unique_skus.join(date_range, how="cross")

master_df = grid_df.join(daily_sales, on=['ItemCode', 'Date'], how='left').with_columns([
    pl.col('Target_Quantity').fill_null(0).cast(pl.Int16),
    pl.col('Returns_Feature').fill_null(0).cast(pl.Int16)
]).sort(['ItemCode', 'Date'])

# Re-attach Prices
price_df = train_df.group_by(['ItemCode', 'Date']).agg(pl.col('UnitPrice').mean().cast(pl.Float32))
master_df = master_df.join(price_df, on=['ItemCode', 'Date'], how='left')
master_df = master_df.with_columns(
    pl.col('UnitPrice').forward_fill().over('ItemCode')
).with_columns(
    pl.col('UnitPrice').backward_fill().over('ItemCode').fill_null(0)
)
print(f"Grid created. Size: {master_df.shape}")

In [ ]:
print("Phase 3: Pre-computing Encodings & Trends")

# Target Encodings
train_slice = master_df.filter(pl.col('Date') <= split_date).with_columns(
    pl.col('Date').dt.weekday().alias('DayOfWeek')
)
day_of_week_mean = train_slice.group_by(['ItemCode', 'DayOfWeek']).agg(
    pl.col('Sales_Lag_28').mean().cast(pl.Float32).alias('Mean_by_DayOfWeek') if 'Sales_Lag_28' in train_slice.columns else pl.lit(0).alias('Mean_by_DayOfWeek')
) 

# Price Tiers (Clustering cheap vs expensive SKUs)
sku_price = master_df.group_by('ItemCode').agg(pl.col('UnitPrice').mean().alias('AvgPrice'))
quantiles = sku_price.select(
    pl.col('AvgPrice').quantile(0.33).alias('q1'),
    pl.col('AvgPrice').quantile(0.67).alias('q2')
).to_dicts()[0]

sku_price = sku_price.with_columns(
    pl.when(pl.col('AvgPrice') < quantiles['q1']).then(pl.lit('Cheap'))
    .when(pl.col('AvgPrice') < quantiles['q2']).then(pl.lit('Medium'))
    .otherwise(pl.lit('Expensive')).alias('PriceTier').cast(pl.Categorical)
)

def create_features(df, is_train=False):
    # Sort guarantees rolling functions work correctly
    df = df.sort(["ItemCode", "Date"])

    # Basic Time
    df = df.with_columns([
        pl.col('Date').dt.weekday().alias('DayOfWeek'),
        pl.col('Date').dt.month().alias('Month'),
        pl.col('Date').dt.year().alias('Year'),
        (pl.col('Date').dt.day() > 25).cast(pl.Int8).alias('Is_Month_End')
    ])
    
    # Lags
    lags = [28, 35, 42, 56]
    df = df.with_columns([pl.col('Target_Quantity').shift(lag).over('ItemCode').alias(f'Sales_Lag_{lag}') for lag in lags])
    
    # Days since last sale
    df = df.with_columns(
        (pl.when(pl.col("Target_Quantity") > 0).then(0).otherwise(1)).cum_sum().over("ItemCode").cast(pl.Int32).alias("days_since_last_sale")
    )
    
    # Zero Streak 
    df = df.with_columns(
        (pl.when(pl.col("Target_Quantity") == 0).then(1).otherwise(0)).cum_sum().over("ItemCode").alias("zero_group")
    )
    df = df.with_columns(
        (pl.when(pl.col("Target_Quantity") == 0)
         .then(pl.col("zero_group").cum_count().over(["ItemCode", "zero_group"]))
         .otherwise(0)).cast(pl.Int32).alias("zero_streak")
    ).drop("zero_group")

    # Rolling means
    rolling_windows = [7, 14, 28]
    df = df.with_columns([
        pl.col('Sales_Lag_28').rolling_mean(window_size=w, min_samples=1).over('ItemCode').cast(pl.Float32).alias(f'Rolling_Mean_28_{w}')
        for w in rolling_windows
    ])
    
    # EWMAs (Exponential Moving Averages)
    df = df.with_columns([
        pl.col("Target_Quantity").shift(1).ewm_mean(span=7).over("ItemCode").cast(pl.Float32).alias("ewm_7"),
        pl.col("Target_Quantity").shift(1).ewm_mean(span=28).over("ItemCode").cast(pl.Float32).alias("ewm_28")
    ])

    # Sale Frequency (How many times it sold in last X days)
    df = df.with_columns([
        ((pl.col("Target_Quantity") > 0).cast(pl.Int8).shift(1).rolling_sum(window_size=28, min_samples=1).over("ItemCode").cast(pl.Float32).alias("sale_frequency_28d")),
        ((pl.col("Target_Quantity") > 0).cast(pl.Int8).shift(1).rolling_sum(window_size=90, min_samples=1).over("ItemCode").cast(pl.Float32).alias("sale_frequency_90d"))
    ])

    # Trend Ratios
    df = df.with_columns([
        ((pl.col("Rolling_Mean_28_7") + 1e-6) / (pl.col("Rolling_Mean_28_28") + 1e-6)).cast(pl.Float32).alias("trend_ratio_7_28"),
        ((pl.col("ewm_7") + 1e-6) / (pl.col("ewm_28") + 1e-6)).cast(pl.Float32).alias("ewm_trend_ratio")
    ])

    # Zero streaking on Lags & Volatility
    df = df.with_columns(
        pl.col('Target_Quantity').shift(56).over('ItemCode').alias('_Anchor_Lag_Real')
    )
    
    df = df.with_columns([
        (pl.col('Sales_Lag_28') == 0).cast(pl.Int32).rolling_sum(window_size=28, min_samples=1).over('ItemCode').cast(pl.Int16).alias('Zero_Count_28'),
        pl.col('_Anchor_Lag_Real').rolling_std(window_size=28, min_samples=2).over('ItemCode').cast(pl.Float32).fill_null(0).alias('Rolling_Std_28_28'),
        pl.col('_Anchor_Lag_Real').rolling_max(window_size=28, min_samples=1).over('ItemCode').cast(pl.Int16).alias('Rolling_Max_28_28')
    ]).drop('_Anchor_Lag_Real')
    
    # Price Momentum
    df = df.with_columns(pl.col('UnitPrice').shift(28).over('ItemCode').alias('Price_Lag_28'))
    df = df.with_columns(((pl.col('UnitPrice') / (pl.col('Price_Lag_28') + 1e-5)).cast(pl.Float32).alias('Price_Momentum')))

    # Holidays (Vietnam Proxy)
    df = df.with_columns([
        pl.when(pl.col('Month').is_in([1, 2])).then(1).otherwise(0).cast(pl.Int8).alias('IsTetHoliday'),
        pl.when(pl.col('Month').is_in([4, 5, 9])).then(1).otherwise(0).cast(pl.Int8).alias('IsNationalHolidayMonth')
    ])

    # Append Pre-Calculated Tiers & Encodings safely
    df = df.with_columns([pl.col('ItemCode').cast(pl.Utf8), pl.col('DayOfWeek').cast(pl.Utf8)])
    
    # Re-calculate DayOfWeek Target encoding strictly on training
    if is_train:
        global day_of_week_mean
        train_slice = df.filter(pl.col('Date') <= split_date)
        day_of_week_mean = train_slice.group_by(['ItemCode', 'DayOfWeek']).agg(
            pl.col('Sales_Lag_28').mean().cast(pl.Float32).alias('Mean_by_DayOfWeek')
        ).with_columns([pl.col('ItemCode').cast(pl.Utf8), pl.col('DayOfWeek').cast(pl.Utf8)])
        
    df = df.join(day_of_week_mean, on=['ItemCode', 'DayOfWeek'], how='left').with_columns(pl.col('Mean_by_DayOfWeek').fill_null(0))

    # Join Price Tiers
    df = df.join(sku_price.select(['ItemCode', 'PriceTier']).with_columns(pl.col('ItemCode').cast(pl.Utf8)), on='ItemCode', how='left')
    
    # Tier Trends
    if is_train:
        global tier_trend
        tier_trend = df.filter(pl.col('Date') <= split_date).group_by(['Date', 'PriceTier']).agg(
            pl.col('Sales_Lag_28').mean().cast(pl.Float32).alias('Tier_Mean_Lag_28')
        )
    df = df.join(tier_trend, on=['Date', 'PriceTier'], how='left').with_columns(pl.col('Tier_Mean_Lag_28').fill_null(0).forward_fill().over('PriceTier'))

    # Final cast for LightGBM
    df = df.with_columns([
        pl.col("ItemCode").cast(pl.Categorical),
        pl.col("DayOfWeek").cast(pl.Categorical),
        pl.col("Month").cast(pl.Categorical),
        pl.col("PriceTier").cast(pl.Categorical)
    ])
    
    return df

master_df = create_features(master_df, is_train=True)
master_df = master_df.drop_nulls(subset=['Sales_Lag_28'])
print("\nDone")

In [ ]:
print("Phase 4: Injecting WRMSSE scale into LightGBM weights")

train_for_scale = master_df.filter(pl.col('Date') <= split_date)
first_sale_df = train_for_scale.filter(pl.col('Target_Quantity') > 0).group_by('ItemCode').agg(pl.col('Date').min().alias('First_Sale_Date'))
train_for_scale = train_for_scale.join(first_sale_df, on='ItemCode', how='inner').filter(pl.col('Date') >= pl.col('First_Sale_Date'))

scale_df = train_for_scale.with_columns(
    (pl.col('Target_Quantity') - pl.col('Target_Quantity').shift(1).over('ItemCode')).pow(2).alias('naive_sq_error')
).drop_nulls(subset=['naive_sq_error'])

scale_df = scale_df.group_by('ItemCode').agg(
    pl.col('naive_sq_error').mean().alias('scale_denominator')
).with_columns(
    pl.when(pl.col('scale_denominator').is_null() | (pl.col('scale_denominator') <= 0))
      .then(1e-5).otherwise(pl.col('scale_denominator')).alias('scale_denominator')
)

weights_for_lgb = weights_df.with_columns(pl.col('ItemCode').cast(pl.Utf8)).join(
    scale_df.with_columns(pl.col('ItemCode').cast(pl.Utf8)), on='ItemCode', how='left'
).with_columns(
    (pl.col('Weight') / pl.col('scale_denominator').sqrt()).alias('LGB_Sample_Weight')
).with_columns(pl.col('LGB_Sample_Weight').fill_null(0))

master_df = master_df.with_columns(pl.col('ItemCode').cast(pl.Utf8)).join(
    weights_for_lgb.select(['ItemCode', 'LGB_Sample_Weight']), on='ItemCode', how='left'
).with_columns(pl.col('LGB_Sample_Weight').fill_null(0).cast(pl.Float32))

In [ ]:
print("Phase 5: Training model")
master_df = master_df.with_columns([
    pl.col("ItemCode").cast(pl.Categorical),
    pl.col("DayOfWeek").cast(pl.Categorical),
    pl.col("Month").cast(pl.Categorical),
    pl.col("PriceTier").cast(pl.Categorical)
])

TARGET = 'Target_Quantity'
FEATURES = [
    'ItemCode', 'DayOfWeek', 'Month', 'Year', 'Is_Month_End', 'Returns_Feature', 
    'Sales_Lag_28', 'Sales_Lag_35', 'Sales_Lag_42', 'Sales_Lag_56',
    'Rolling_Mean_28_7', 'Rolling_Mean_28_14', 'Rolling_Mean_28_28',
    'Zero_Count_28', 'Rolling_Std_28_28', 'Rolling_Max_28_28', 'Mean_by_DayOfWeek',
    'IsTetHoliday', 'IsNationalHolidayMonth', 'UnitPrice', 'Price_Lag_28', 'Price_Momentum',
    'PriceTier', 'Tier_Mean_Lag_28', 'days_since_last_sale', 'zero_streak','ewm_7', 'ewm_28',
    'sale_frequency_28d', 'sale_frequency_90d', 'trend_ratio_7_28', 'ewm_trend_ratio'
]
CATEGORICAL_FEATURES = ['ItemCode', 'DayOfWeek', 'Month', 'PriceTier']

train_data = master_df.filter(pl.col('Date') <= split_date)
valid_data = master_df.filter(pl.col('Date') > split_date)
valid_meta = valid_data.select(['ItemCode', 'Date', 'Target_Quantity'])

X_train = train_data.select(FEATURES).to_pandas()
y_train = train_data.select(TARGET).to_pandas()[TARGET]
X_valid = valid_data.select(FEATURES).to_pandas()
y_valid = valid_data.select(TARGET).to_pandas()[TARGET]

train_weight = train_data.select('LGB_Sample_Weight').to_pandas()['LGB_Sample_Weight'].values
valid_weight = valid_data.select('LGB_Sample_Weight').to_pandas()['LGB_Sample_Weight'].values

lgb_train = lgb.Dataset(X_train, label=y_train, weight=train_weight, categorical_feature=CATEGORICAL_FEATURES)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, weight=valid_weight, categorical_feature=CATEGORICAL_FEATURES, reference=lgb_train)

params = {
    'objective': 'tweedie',          
    'tweedie_variance_power': 1.30,  
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,           
    'num_leaves': 31,
    'min_data_in_leaf': 200,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'n_jobs': -1,
    'seed': 42,
    'verbose': -1
}

callbacks = [lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=100)]

model = lgb.train(
    params, lgb_train, num_boost_round=3000, valid_sets=[lgb_train, lgb_valid], callbacks=callbacks
)

# Score Local Validation
preds = model.predict(X_valid, num_iteration=model.best_iteration)
valid_results = valid_meta.with_columns(pl.Series(name="Predicted_Quantity", values=preds)).with_columns(pl.when(pl.col('Predicted_Quantity') < 0).then(0).otherwise(pl.col('Predicted_Quantity')))
valid_results = valid_results.with_columns(pl.Series(name="Zero_Count_28", values=X_valid['Zero_Count_28']))

# Dynamic multiplier logic
valid_results = valid_results.with_columns(
    pl.when(pl.col('Zero_Count_28') == 28).then(0)                               
    .when(pl.col('Zero_Count_28') >= 25).then((pl.col('Predicted_Quantity') * 0.80).round(0))  
    .when(pl.col('Zero_Count_28') >= 15).then((pl.col('Predicted_Quantity') * 0.95).round(0))  
    .otherwise(pl.col('Predicted_Quantity').round(0))                            
    .cast(pl.Int16).alias('Predicted_Quantity')
)

def calculate_local_wrmsse(df, valid_res, weights):
    train_grid = df.filter(pl.col('Date') <= split_date)
    f_sale = train_grid.filter(pl.col('Target_Quantity') > 0).group_by('ItemCode').agg(pl.col('Date').min().alias('First_Sale_Date'))
    train_grid = train_grid.join(f_sale, on='ItemCode', how='inner').filter(pl.col('Date') >= pl.col('First_Sale_Date'))
    scale = train_grid.with_columns((pl.col('Target_Quantity') - pl.col('Target_Quantity').shift(1).over('ItemCode')).pow(2).alias('naive_sq_error')).drop_nulls(subset=['naive_sq_error'])
    scale = scale.group_by('ItemCode').agg(pl.col('naive_sq_error').mean().alias('scale_denominator')).with_columns(
        pl.when(pl.col('scale_denominator').is_null() | (pl.col('scale_denominator') <= 0)).then(1e-5).otherwise(pl.col('scale_denominator')).alias('scale_denominator')
    )
    val_mse = valid_res.with_columns((pl.col('Target_Quantity') - pl.col('Predicted_Quantity')).pow(2).alias('model_sq_error')).group_by('ItemCode').agg(pl.col('model_sq_error').mean().alias('model_mse')).with_columns(pl.col('model_mse').fill_null(0))
    metrics = val_mse.join(scale, on='ItemCode', how='left').with_columns(pl.col('scale_denominator').fill_null(1e-5)).with_columns((pl.col('model_mse') / pl.col('scale_denominator')).sqrt().alias('RMSSE')).with_columns(pl.col('RMSSE').fill_nan(0).fill_null(0))
    weights = weights.with_columns(pl.col('ItemCode').cast(pl.Categorical))
    final = metrics.join(weights, on='ItemCode', how='left').with_columns([pl.col('Weight').fill_null(0), pl.col('RMSSE').fill_null(0)]).with_columns((pl.col('RMSSE') * pl.col('Weight')).alias('Weighted_RMSSE'))
    return final['Weighted_RMSSE'].fill_nan(0).fill_null(0).sum()

local_score = calculate_local_wrmsse(master_df, valid_results, weights_df)
print(f"\nLOCAL WRMSSE SCORE: {local_score:.5f}")

In [ ]:
print("Phase 6: Generating future predictions")

val_start = datetime.date(2025, 9, 6)
eval_end = datetime.date(2025, 10, 31)

history_cutoff = val_start - datetime.timedelta(days=365)
print(f"   [+] Slicing history from {history_cutoff} to prevent memory overload...")

# Grab latest prices to drag into the future
latest_price = master_df.group_by('ItemCode').agg(pl.col('UnitPrice').last()).with_columns(pl.col('ItemCode').cast(pl.Utf8))

# Filter history to just the last 365 days
raw_history = master_df.filter(pl.col('Date') >= history_cutoff).select(
    ['ItemCode', 'Date', 'Target_Quantity', 'Returns_Feature', 'UnitPrice']
).with_columns(pl.col('ItemCode').cast(pl.Utf8))

# Spawn future grid
future_dates = pl.date_range(val_start, eval_end, interval="1d", eager=True).alias("Date").to_frame()
unique_skus = raw_history.select('ItemCode').unique()

future_grid = unique_skus.join(future_dates, how="cross").with_columns([
    pl.lit(0).cast(pl.Int16).alias('Target_Quantity'),
    pl.lit(0).cast(pl.Int16).alias('Returns_Feature')
])
future_grid = future_grid.join(latest_price, on='ItemCode', how='left').with_columns(pl.col('UnitPrice').fill_null(0))

# Combine to create the inference dataset
full_raw = pl.concat([raw_history, future_grid]).sort(['ItemCode', 'Date'])

print("Predicting Validation")
full_features = create_features(full_raw, is_train=False) # Make sure this matches your function name!
val_subset = full_features.filter((pl.col('Date') >= val_start) & (pl.col('Date') <= datetime.date(2025, 10, 3)))

val_preds = model.predict(val_subset.select(FEATURES).to_pandas(), num_iteration=model.best_iteration)

val_subset = val_subset.with_columns([
    pl.Series(name="Raw_Pred", values=val_preds), 
    pl.col('ItemCode').cast(pl.Utf8)
]).with_columns(
    pl.when(pl.col('Raw_Pred') < 0).then(0).otherwise(pl.col('Raw_Pred'))
)

# Apply dynamic multipliers from our Grid Search optimization
val_subset = val_subset.with_columns(
    pl.when(pl.col('Zero_Count_28') == 28).then(0)                               
    .when(pl.col('Zero_Count_28') >= 25).then((pl.col('Raw_Pred') * 0.80).round(0))  
    .when(pl.col('Zero_Count_28') >= 15).then((pl.col('Raw_Pred') * 0.95).round(0))  
    .otherwise(pl.col('Raw_Pred').round(0)).cast(pl.Int16).alias('Pred')
)

# Overwrite the empty future dates with Phase 1 predictions so Phase 2 can use them as history
full_raw = full_raw.join(val_subset.select(['ItemCode', 'Date', 'Pred']), on=['ItemCode', 'Date'], how='left')
full_raw = full_raw.with_columns(
    pl.when(pl.col('Pred').is_not_null()).then(pl.col('Pred'))
    .otherwise(pl.col('Target_Quantity')).cast(pl.Int16).alias('Target_Quantity')
).drop('Pred')

print("Predicting Evaluation")
full_features_phase2 = create_features(full_raw, is_train=False)
eval_subset = full_features_phase2.filter(pl.col('Date') >= datetime.date(2025, 10, 4))

eval_preds = model.predict(eval_subset.select(FEATURES).to_pandas(), num_iteration=model.best_iteration)

eval_subset = eval_subset.with_columns([
    pl.Series(name="Raw_Pred", values=eval_preds), 
    pl.col('ItemCode').cast(pl.Utf8)
]).with_columns(
    pl.when(pl.col('Raw_Pred') < 0).then(0).otherwise(pl.col('Raw_Pred'))
)

eval_subset = eval_subset.with_columns(
    pl.when(pl.col('Zero_Count_28') == 28).then(0)                               
    .when(pl.col('Zero_Count_28') >= 25).then((pl.col('Raw_Pred') * 0.80).round(0))  
    .when(pl.col('Zero_Count_28') >= 15).then((pl.col('Raw_Pred') * 0.95).round(0))  
    .otherwise(pl.col('Raw_Pred').round(0)).cast(pl.Int16).alias('Pred')
)

print("Formatting Submission CSV")

val_final = val_subset.select(['ItemCode', 'Date', 'Pred']).rename({'Pred': 'Predicted_Quantity'})
eval_final = eval_subset.select(['ItemCode', 'Date', 'Pred']).rename({'Pred': 'Predicted_Quantity'})

# Pivot Validation data
val_pivot = val_final.pivot(values='Predicted_Quantity', index='ItemCode', on='Date')
val_cols = sorted([col for col in val_pivot.columns if col != 'ItemCode'])
val_pivot = val_pivot.rename({col: f'F{i+1}' for i, col in enumerate(val_cols)})
val_pivot = val_pivot.with_columns((pl.col('ItemCode') + '_validation').alias('id')).drop('ItemCode')

# Pivot Evaluation data
eval_pivot = eval_final.pivot(values='Predicted_Quantity', index='ItemCode', on='Date')
eval_cols = sorted([col for col in eval_pivot.columns if col != 'ItemCode'])
eval_pivot = eval_pivot.rename({col: f'F{i+1}' for i, col in enumerate(eval_cols)})
eval_pivot = eval_pivot.with_columns((pl.col('ItemCode') + '_evaluation').alias('id')).drop('ItemCode')

# Stack them together
final_sub = pl.concat([val_pivot, eval_pivot], how="vertical").with_columns(pl.col('id').cast(pl.Utf8))

# Map to the official sample submission file structure to guarantee 0 errors
if os.path.exists(sample_sub_path):
    sample_df = pl.read_csv(sample_sub_path).with_columns(pl.col('id').cast(pl.Utf8))
    submission_ready = sample_df.select('id').join(final_sub, on='id', how='left').fill_null(0)
else:
    submission_ready = final_sub.fill_null(0).select(['id'] + [f'F{i}' for i in range(1, 29)])

# Ensure absolute integer format before submitting
for i in range(1, 29):
    submission_ready = submission_ready.with_columns(pl.col(f'F{i}').cast(pl.Int32))

file_name = f'submission_final_{datetime.datetime.now().strftime("%H%M")}.csv'
submission_ready.write_csv(file_name)

print(f"\nPrediction file generated: {file_name}")